## API Data Analysis Using Python Structures

## Project Overview

**Pesa Pulse**, a Nairobi fintech start-up, has just signed a partner who shares its customer directory through an API. Before the marketing team plans regional campaigns, they want to know the basics: how many customers there are, where they are, and what the data even looks like.

You are the junior analyst. Pull the data, explore its nested structure, extract the attributes that matter, and answer "where are our customers?" first with plain Python structures and then with pandas. Finish with a chart marketing can drop straight into a slide.

### Objectives

- Retrieve JSON from an API with requests and handle a failed request
- Navigate nested dictionaries and lists confidently
- Use sets to find unique values and dictionaries to count them
- Write dict comprehensions for aggregation without pandas
- Move the cleaned data into pandas, export it and visualise a distribution

## Task 1 — Retrieve Customer Data Using an API

 Created  function `get_customers()`

In [1]:

import requests

def get_customers(url):
    
    try:
        # Send a request to the API and wait for at most 10 seconds.
        response = requests.get(url, timeout=10)

        # Raise an error if the server returns an unsuccessful status.
        response.raise_for_status()

        # Convert the JSON response into Python objects.
        customers = response.json()

        # Confirm that the API returned a list.
        if not isinstance(customers, list):
            print("Error: Expected a list of customers.")
            return []

        return customers

    except requests.exceptions.RequestException as error:
        # Handle connection, timeout and HTTP errors.
        print(f"Failed to retrieve customers: {error}")
        return []


# API endpoint
url = "https://jsonplaceholder.typicode.com/users"

# Retrieve the data
customers = get_customers(url)

# Display the number of records retrieved
print("Total customers:", len(customers))


Failed to retrieve customers: HTTPSConnectionPool(host='jsonplaceholder.typicode.com', port=443): Max retries exceeded with url: /users (Caused by NameResolutionError("HTTPSConnection(host='jsonplaceholder.typicode.com', port=443): Failed to resolve 'jsonplaceholder.typicode.com' ([Errno 11001] getaddrinfo failed)"))
Total customers: 0


## Task 2 — Explore Nested Dictionaries and Lists


To explore the structure recursively and display every key and its value's Python type, I created a function  `explore()`

In [2]:

def explore(record, indent=0):
    """Print dictionary keys and Python types at each nesting level."""

    # Create spaces to show the nesting level.
    prefix = " " * indent

    # If the record is a dictionary, inspect its keys and values.
    if isinstance(record, dict):
        for key, value in record.items():

            # Print the key and its value's Python type.
            print(f"{prefix}{key}: {type(value).__name__}")

            # Explore nested dictionaries and lists.
            if isinstance(value, (dict, list)):
                explore(value, indent + 2)

    # If the record is a list, inspect each item.
    elif isinstance(record, list):
        for index, value in enumerate(record):
            print(f"{prefix}[{index}]: {type(value).__name__}")

            if isinstance(value, (dict, list)):
                explore(value, indent + 2)


# Explore the first customer.
if customers:
    explore(customers[0])


## Task 3 — Extract Customer Attributes

Created a function, `extract_attributes()` to flatten nested customer records into simple dictionaries

In [3]:

def extract_attributes(customers):
    """Flatten nested customer records into simple dictionaries."""

    rows = []

    for customer in customers:

        # Extract nested dictionaries.
        address = customer.get("address", {})
        geo = address.get("geo", {})
        company = customer.get("company", {})

        # Extract the email domain.
        email = customer.get("email", "")
        email_domain = email.rsplit("@", 1)[-1].lower() if "@" in email else ""

        # Convert coordinates into floats.
        latitude = float(geo["lat"]) if geo.get("lat") else float("nan")
        longitude = float(geo["lng"]) if geo.get("lng") else float("nan")

        # Create one flat dictionary per customer.
        row = {
            "id": customer.get("id"),
            "name": customer.get("name"),
            "username": customer.get("username"),
            "email": email,
            "email_domain": email_domain,
            "phone": customer.get("phone"),
            "city": address.get("city"),
            "zipcode": address.get("zipcode"),
            "latitude": latitude,
            "longitude": longitude,
            "company": company.get("name"),
            "website": customer.get("website")
        }

        rows.append(row)

    return rows


# Extract the attributes.
rows = extract_attributes(customers)

# Inspect the first flat record.
if rows:
    print(rows[0])


## Task 4 — Use Sets to Find Unique Cities

Created a function  `unique_cities()` to return the unique cities in the customer records

In [4]:

def unique_cities(customers):
    """Return the unique cities in the customer records."""

    cities = {
        customer["address"]["city"]
        for customer in customers
        if customer.get("address", {}).get("city")
    }

    return cities


# Find unique cities.
cities = unique_cities(customers)

print("Number of customers:", len(customers))
print("Number of unique cities:", len(cities))
print("Unique cities:", sorted(cities))


Number of customers: 0
Number of unique cities: 0
Unique cities: []


## Task 5 — Count Customers by City and Email Domain

### Part A: Count Customers by City Using a Loop

Created a function `city_counts()` to count the number of customers in each city

In [5]:

def city_counts(customers):
    """Count the number of customers in each city."""

    counts = {}

    for customer in customers:
        city = customer.get("address", {}).get("city")

        if city:
            # Add one to the existing count, or start at zero.
            counts[city] = counts.get(city, 0) + 1

    return counts


counts = city_counts(customers)

# Sort from most common to least common.
sorted_counts = dict(
    sorted(counts.items(), key=lambda item: item[1], reverse=True)
)

print("Customers by city:")
print(sorted_counts)


Customers by city:
{}


### Part B: Count Customers by Email Domain Using a Dictionary Comprehension

Created a function email_domain_counts to count email domains using a dictionary comprehension

In [6]:

def email_domain_counts(customers):
    """Count email domains using a dictionary comprehension."""

    domains = [
        customer["email"].rsplit("@", 1)[-1].lower()
        for customer in customers
        if "@" in customer.get("email", "")
    ]

    # Count each distinct domain using a dictionary comprehension.
    counts = {
        domain: domains.count(domain)
        for domain in set(domains)
    }

    return counts


domain_counts = email_domain_counts(customers)

# Sort from most common to least common.
sorted_domain_counts = dict(
    sorted(
        domain_counts.items(),
        key=lambda item: item[1],
        reverse=True
    )
)

print("Customers by email domain:")
print(sorted_domain_counts)


Customers by email domain:
{}


## Task 6 — Convert the Records into a Pandas DataFrame

In [7]:

import pandas as pd

def to_dataframe(rows):
    """Convert a list of dictionaries into a DataFrame."""
    return pd.DataFrame(rows)


df = to_dataframe(rows)

# Print the number of rows and columns.
print("DataFrame shape:", df.shape)

# Display column names and data types.
print("\nData types:")
print(df.dtypes)

# Count customers by city using pandas.
pandas_city_counts = df["city"].value_counts().to_dict()

print("\nPandas city counts:")
print(pandas_city_counts)

# Compare pandas results with our Python dictionary.
print("\nDo the counts match?")
print(pandas_city_counts == counts)


DataFrame shape: (0, 0)

Data types:
Series([], dtype: object)


KeyError: 'city'

## Task 7 — Export the Results to CSV

In [ ]:

from pathlib import Path
import pandas as pd

# Create the output folder if it does not exist.
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

# Export customer records without the DataFrame index.
df.to_csv(output_dir / "customers.csv", index=False)

# Convert the city-count dictionary into a DataFrame.
city_counts_df = pd.DataFrame(
    sorted_counts.items(),
    columns=["city", "customer_count"]
)

# Export the city counts without the index.
city_counts_df.to_csv(
    output_dir / "city_counts.csv",
    index=False
)

print("CSV files exported successfully.")


CSV files exported successfully.


## Task 8 — Visualize the Customer Distribution


In [ ]:

import matplotlib.pyplot as plt
from pathlib import Path

def plot_distribution(counts, path):
    """Create and save a horizontal bar chart."""

    if not counts:
        print("No data available for the chart.")
        return

    # Sort the categories from most common to least common.
    sorted_items = sorted(
        counts.items(),
        key=lambda item: item[1],
        reverse=True
    )

    labels = [item[0] for item in sorted_items]
    values = [item[1] for item in sorted_items]

    # Create the chart.
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(labels[::-1], values[::-1])

    ax.set_title("Customer Distribution")
    ax.set_xlabel("Number of customers")
    ax.set_ylabel("Category")

    fig.tight_layout()

    # Create the parent folder if needed and save the chart.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Chart saved to: {path}")


# Create both charts.
plot_distribution(
    sorted_counts,
    "output/customer_distribution.png"
)

plot_distribution(
    sorted_domain_counts,
    "output/email_domain_distribution.png"
)


Chart saved to: output\customer_distribution.png
Chart saved to: output\email_domain_distribution.png
